# 02 — Preference Alignment with DPO
### Stage 3: turning the SFT model into "PostTraining Tutor - v2"

This notebook loads the SFT adapter from Notebook 1 and further trains it with **Direct Preference Optimization (DPO)** on `preference_dataset.jsonl`, so the model learns to prefer better explanations over worse ones for the same question.


## Step 0 — Install dependencies

**What/Why:** same reasoning as Notebook 1 - Unsloth for fast LoRA training, TRL for `DPOTrainer`.

In [1]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" "transformers==4.57.1" "trl==0.20.0" "peft>=0.19.1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.3 MB/s e

In [2]:
# WHAT: imports for DPO training.
import torch
import json
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import DPOTrainer, DPOConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Step 1 — Load the SFT model (base + SFT adapter)

**What:** reload the same base model and re-attach the LoRA adapter saved at the end of Notebook 1.
**Why:** DPO in this project is applied *on top of* the SFT model - we are refining an already instruction-tuned model's preferences, not starting from the raw base model.
**How:** load the base 4-bit model exactly as before, then load the saved adapter directory as the starting LoRA weights (instead of creating a fresh, randomly-initialized adapter).

In [3]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

# Your SFT adapter on Hugging Face
SFT_ADAPTER = "Hiteshwari7/posttraining-tutor-sft-adapter"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Load the SFT adapter as TRAINABLE
model.load_adapter(
    SFT_ADAPTER,
    adapter_name="default",
    is_trainable=True,
)

# Make SFT adapter active
model.set_adapter("default")

# Switch to training mode
FastLanguageModel.for_training(model)

# Verify trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

==((====))==  Unsloth 2026.8.7: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

Trainable parameters: 24,313,856
Total parameters: 1,827,777,536
Trainable %: 1.3302%


## Step 2 — Prepare the preference dataset

**What:** load `3.preference_dataset.jsonl`, where each row has `prompt`, `chosen`, and `rejected` fields.
**Why:** DPO needs *paired* responses to the same prompt - one preferred, one not - so it can directly increase the model's relative probability of the chosen response versus the rejected one.
**How:** `DPOTrainer` expects a dataset with exactly these three columns (as raw text, not yet tokenized) - it handles tokenization and log-probability computation internally.

In [5]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

pref_rows = load_jsonl("3. preference_dataset.jsonl")
print(f"Loaded {len(pref_rows)} preference pairs")
print(pref_rows[0])

# WHAT: wrap each prompt in the chat template (system + user turn), leaving
#       chosen/rejected as plain assistant-response text - DPOTrainer appends
#       them to the templated prompt internally when computing log-probs.
def format_prompt(example):
    messages = [
        {"role": "system", "content": "You are PostTraining Tutor, an assistant that explains LLM training concepts clearly and concisely."},
        {"role": "user", "content": example["prompt"]},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {
        "prompt": prompt_text,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

dpo_dataset = Dataset.from_list(pref_rows).map(format_prompt)


Loaded 60 preference pairs
{'prompt': 'What is LoRA?', 'chosen': 'LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method where the original model weights are frozen and small trainable low-rank matrices are added. These adapters learn task-specific changes while requiring far fewer trainable parameters.', 'rejected': 'LoRA is a technique that retrains every parameter of a language model from scratch so the entire model can learn new information.'}


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

## Step 3 — Configure DPOTrainer

**What DPO optimizes:** for each (prompt, chosen, rejected) triple, DPO increases the model's log-probability of `chosen` relative to `rejected`, relative to a frozen reference model - all in a single supervised-style loss, with **no separate reward model and no RL rollout loop** (unlike classic RLHF/PPO).

**Key parameters explained:**
- `beta=0.1`: controls how strongly the model is pushed away from the reference (SFT) model's behavior. Lower `beta` -> larger, more aggressive preference updates; higher `beta` -> more conservative, stays closer to the SFT model. `0.1`-`0.5` is a typical starting range.
- `learning_rate=5e-6`: DPO learning rates are usually much lower than SFT's - we are fine-tuning *preferences* on top of an already-good model, and a large LR here can quickly destabilize fluency.
- `num_train_epochs=1-2`: preference datasets are typically small; too many epochs risks overfitting to the specific chosen/rejected pairs rather than learning the general preference pattern.
- **Reference model**: because we're using PEFT/LoRA, `DPOTrainer` can use the *same* underlying base model with the adapter disabled as the implicit reference - no need to keep a second full copy of the model in memory (a big practical win for Colab).

In [6]:
from trl import DPOTrainer, DPOConfig
import torch

# Prepare model for training
FastLanguageModel.for_training(model)

# Reduce memory usage
model.config.use_cache = False

# Ensure trainable LoRA params are fp32
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype == torch.float16:
        param.data = param.data.float()

dpo_args = DPOConfig(
    output_dir="outputs/stage3_dpo",

    beta=0.1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=2,

    learning_rate=5e-6,

    logging_steps=5,

    optim="adamw_torch",

    warmup_steps=5,
    lr_scheduler_type="linear",

    max_length=1024,

    seed=42,
    report_to="none",

    fp16=True,
    bf16=False,
)

dpo_trainer = DPOTrainer(
    model=model,

    # IMPORTANT: no separate reference model
    ref_model=None,

    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

dpo_trainer.train()

Extracting prompt in train dataset (num_proc=2):   0%|          | 0/60 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/60 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 2 | Total steps = 16
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.688500,0.000272,-0.009022,0.775000,0.009294,-61.373005,-55.221199,0.326863,0.189803
10,0.638300,0.009336,-0.106812,1.000000,0.116147,-59.979935,-55.061436,0.415700,0.203434
15,0.587300,0.021891,-0.203012,1.000000,0.224903,-62.273659,-56.457237,0.297572,0.206404


TrainOutput(global_step=16, training_loss=0.6348637267947197, metrics={'train_runtime': 91.5512, 'train_samples_per_second': 1.311, 'train_steps_per_second': 0.175, 'total_flos': 0.0, 'train_loss': 0.6348637267947197, 'epoch': 2.0})

In [16]:
import torch

current_state = model.state_dict()

max_diff = 0.0
different_tensors = 0
checked = 0

for sft_key, sft_tensor in sft_weights.items():

    # Convert:
    # base_model.model.model.layers...lora_A.weight
    # ->
    # model.layers...lora_A.default.weight

    dpo_key = sft_key

    if dpo_key.startswith("base_model.model."):
        dpo_key = dpo_key[len("base_model.model."):]

    dpo_key = dpo_key.replace(
        ".lora_A.weight",
        ".lora_A.default.weight"
    ).replace(
        ".lora_B.weight",
        ".lora_B.default.weight"
    )

    if dpo_key not in current_state:
        continue

    # Put both tensors on CPU before comparing
    sft_tensor = sft_tensor.detach().float().cpu()
    dpo_tensor = current_state[dpo_key].detach().float().cpu()

    diff = torch.max(torch.abs(sft_tensor - dpo_tensor)).item()

    max_diff = max(max_diff, diff)

    if diff > 0:
        different_tensors += 1

    checked += 1

print("SFT tensors checked:", checked)
print("Different tensors:", different_tensors)
print("Maximum SFT vs DPO difference:", max_diff)

if checked > 0:
    print("All weights identical:", max_diff == 0.0)
else:
    print("NO TENSORS WERE COMPARED — comparison failed.")

SFT tensors checked: 392
Different tensors: 392
Maximum SFT vs DPO difference: 5.473196506500244e-05
All weights identical: False


## Step 4 — Save the DPO-aligned adapter

**What/Why:** same reasoning as Notebook 1 - save only the (now further-updated) LoRA adapter, this time under a separate directory so we can still load the SFT-only version independently in evaluation.

In [17]:
DPO_ADAPTER_DIR = "outputs/dpo_adapter"

model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)

model.push_to_hub(
    "Hiteshwari7/postraining-tutor-dpo-adapter"
)

tokenizer.push_to_hub(
    "Hiteshwari7/postraining-tutor-dpo-adapter"
)

print("Uploaded successfully!")




Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  16%|#6        | 16.0MB / 97.3MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/Hiteshwari7/postraining-tutor-dpo-adapter


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp3g97vlum/tokenizer.json:  92%|#########2| 15.9MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded successfully!


## Step 5 — Inference comparison: SFT vs. DPO

**What:** ask the same questions and compare the SFT-only adapter's answers to the DPO-aligned adapter's answers.
**Why:** this is the most direct, human-readable evidence of what DPO changed - look for differences in conciseness, confidence, or hedging.

In [18]:
FastLanguageModel.for_inference(model)  # currently holds the DPO adapter

test_questions = [
    "What is LoRA and why is it useful for fine-tuning large language models?",
    "Explain the difference between SFT and DPO in simple terms.",
    "What does RLHF stand for and how does it relate to DPO?",
]

def generate(model, question):
    messages = [
        {"role": "system", "content": "You are PostTraining Tutor, an assistant that explains LLM training concepts clearly and concisely."},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    output = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== DPO model answers ===")
for q in test_questions:
    print(f"Q: {q}\nA: {generate(model, q)}\n{'-'*80}")

# NOTE: to see the SFT-only answers for a true side-by-side, either:
#   (a) disable the adapter temporarily: `model.disable_adapters()` then re-enable, or
#   (b) reload the SFT adapter from Notebook 1 in a fresh session.
# Notebook 3 does this properly by loading all three checkpoints at once.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== DPO model answers ===
Q: What is LoRA and why is it useful for fine-tuning large language models?
A: LoRA (Low-Rank Adaptation) is a method to fine-tune large language models without retraining the entire model. It works by adapting a smaller lower-rank matrix that represents the adaptation parameters, rather than the full model weights. This approach can significantly reduce training time while preserving the knowledge learned by the original model.
--------------------------------------------------------------------------------
Q: Explain the difference between SFT and DPO in simple terms.
A: SFT (Supervised Fine-Tuning) trains a model on labeled data by predicting the correct output. DPO (Denormalized PreTraining) trains a model on unlabeled text, predicting the next word in a sequence, without labels.
--------------------------------------------------------------------------------
Q: What does RLHF stand for and how does it relate to DPO?
A: RLHF stands for Reinforcement Learni

In [19]:
import gradio as gr

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are PostTraining Tutor, an assistant that explains "
    "LLM training concepts clearly and concisely."
)

def chat(question):
    if not question.strip():
        return "Please enter a question."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True,
    )

    return response


demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Ask anything about LoRA, QLoRA, SFT, DPO, RLHF...",
        label="Your Question",
    ),
    outputs=gr.Textbox(
        lines=10,
        label="PostTraining Tutor",
    ),
    title="PostTraining Tutor (DPO)",
    description=(
        "Ask questions about Post-Training, LoRA, QLoRA, SFT, DPO, RLHF, "
        "Transformers, and LLM fine-tuning."
    ),
    examples=[
        ["What is LoRA?"],
        ["Explain DPO in simple words."],
        ["Difference between SFT and DPO"],
        ["What is RLHF?"],
        ["How does QLoRA reduce memory usage?"],
    ],
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1f6eed3fb1c41dec77.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
